# Clase 8: Treemaps y tablas dinámicas (pivot charts)

**Aviso honesto antes de empezar:** esta clase es más breve que las anteriores. El material fuente de este tema es una guía conceptual (un PDF), sin un notebook de ejercicio propio ni un caso de negocio real del curso —a diferencia de, por ejemplo, el dataset de aerolíneas que usaste en la clase anterior. Aun así, vale la pena que conozcas estas dos herramientas: amplían tu caja de herramientas para dos situaciones muy concretas que los gráficos que ya conoces no cubren bien.

Un **treemap** te sirve cuando tienes datos jerárquicos (categorías dentro de categorías) y quieres ver de un vistazo qué parte pesa más. Una **tabla dinámica (pivot chart)** te sirve cuando tienes una tabla larga y desordenada y necesitas resumirla cruzando dos variables a la vez.

## 1. Treemaps: la analogía del armario

Piensa en un armario con cajones, y dentro de cada cajón, compartimentos.

- El **armario completo** es tu dataset.
- Cada **cajón** es una categoría (por ejemplo, "Electrónica").
- Cada **compartimento** dentro del cajón es una subcategoría (por ejemplo, "Laptops").

El tamaño de cada compartimento te dice, de un vistazo, cuánto espacio ocupa esa categoría dentro del total —sin necesidad de leer ni un solo número.

Un treemap representa datos jerárquicos con rectángulos anidados: cada rama de la jerarquía es un rectángulo, dividido a su vez en rectángulos más pequeños para sus subramas. Se usa mucho en analítica de negocio (ventas por categoría y subcategoría), finanzas (composición de una cartera), o gestión de archivos (espacio ocupado por carpeta).

Lo construyes con `px.treemap()`. Necesitas un dataframe con al menos tres columnas: la categoría, la subcategoría, y un valor numérico que determina el tamaño de cada rectángulo.

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Datos de ejemplo: ventas por categoría y subcategoría
data = {
    'Category': ['Electronics', 'Electronics', 'Electronics',
                 'Furniture', 'Furniture', 'Furniture',
                 'Clothing', 'Clothing', 'Clothing'],
    'Subcategory': ['Laptops', 'Smartphones', 'Tablets',
                     'Chairs', 'Tables', 'Sofas',
                     'Men', 'Women', 'Kids'],
    'Sales': [120000, 80000, 30000,
              50000, 40000, 20000,
              70000, 90000, 40000]
}
df = pd.DataFrame(data)
df

In [ ]:
# Creamos el treemap
fig = px.treemap(
    df,
    path=['Category', 'Subcategory'],  # Define la jerarquía: primero categoría, luego subcategoría
    values='Sales',                    # El tamaño de cada rectángulo
    title='Sales Data Treemap'
)
fig.show()

El parámetro `path` es la clave: define el orden de la jerarquía, de lo más general a lo más específico. Aquí, cada categoría ("Electronics", "Furniture", "Clothing") se convierte en un rectángulo grande, y dentro de él aparecen rectángulos más pequeños para cada subcategoría, con el tamaño proporcional a `Sales`. De un vistazo ves que "Women" es la subcategoría con más ventas dentro de "Clothing", sin leer ni una cifra.

## 2. Tablas dinámicas (pivot tables) con pandas: la analogía de Excel

Si alguna vez usaste tablas dinámicas en Excel (arrastrar un campo a "filas", otro a "columnas" y otro a "valores"), un pivot chart en Python es exactamente eso, pero programático: en vez de arrastrar con el mouse, escribes una línea de código que le dice a pandas cómo cruzar tus columnas.

Antes de graficar, necesitas resumir los datos. Para eso usas `pivot_table()` de pandas, que reorganiza una tabla larga cruzando una o más columnas.

Vamos a simular ventas de productos IT por trimestre, con columnas `Date`, `Category`, `Subcategory` y `Sales` (tal como en el ejemplo del PDF).

In [ ]:
# Datos de ejemplo: ventas de productos IT por trimestre
np.random.seed(42)

quarters = ['Q1', 'Q2', 'Q3', 'Q4']
categories = ['Desktops', 'Laptops', 'Peripherals', 'Software']
subcategories = ['Accessories', 'Components', 'Software Suites']

n = 200
df_it = pd.DataFrame({
    'Date': np.random.choice(quarters, n),
    'Category': np.random.choice(categories, n),
    'Subcategory': np.random.choice(subcategories, n),
    'Sales': np.random.randint(1000, 5000, n)
})
df_it.head()

In [ ]:
# Creamos la tabla dinámica:
# - index: qué va en las filas (aquí, el trimestre)
# - columns: qué va en las columnas (aquí, categoría y subcategoría)
# - values: qué columna se agrega
# - aggfunc: cómo se agrega (aquí, suma)
pivot_table = df_it.pivot_table(
    index='Date',
    columns=['Category', 'Subcategory'],
    values='Sales',
    aggfunc='sum'
)
pivot_table

El resultado es una tabla donde cada fila es un trimestre y cada columna es una combinación de categoría y subcategoría, con la suma de ventas correspondiente. Es exactamente lo que harías en Excel arrastrando campos a las zonas de filas, columnas y valores.

## 3. De la tabla dinámica al gráfico (pivot chart)

Una vez tienes la tabla dinámica, graficarla es directo: pandas te deja llamar `.plot()` sobre ella.

In [ ]:
# Graficamos la tabla dinámica como barras
pivot_table.plot(kind='bar', figsize=(14, 8))
plt.title('Sales Summary of IT Products by Category and Subcategory')
plt.xlabel('Quarters')
plt.ylabel('Total Sales')
plt.grid(False)
plt.legend(title=('Category', 'Subcategory'), bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Cada barra agrupada representa un trimestre, y dentro de cada grupo hay una barra por cada combinación de categoría y subcategoría. `bbox_to_anchor` y `loc` en `plt.legend()` sacan la leyenda fuera del área del gráfico, para que no tape las barras cuando hay muchas combinaciones.

## Puntos clave

- Un treemap muestra jerarquías (categoría → subcategoría) con rectángulos anidados, donde el tamaño comunica la magnitud sin necesidad de leer números.
- En `px.treemap()`, el parámetro `path` define el orden de la jerarquía (de general a específico) y `values` define el tamaño de cada rectángulo.
- Una tabla dinámica (`pivot_table()`) resume una tabla larga cruzando filas (`index`), columnas (`columns`) y un valor agregado (`values` + `aggfunc`) — es la versión programática de las tablas dinámicas de Excel.
- Una vez tienes la tabla dinámica, `.plot(kind='bar')` la convierte directamente en un gráfico de barras agrupadas.
- Este par de herramientas no reemplaza lo que ya sabes (scatter, bar, pie...): las usas específicamente cuando tus datos son jerárquicos o cuando necesitas cruzar dos variables categóricas a la vez.